# Step 2: GTFS Snapshot Selection & Active Calendar Expansion

## Objectives
1. Map target service dates (`2025-11-23` to `2026-07-31`) to their corresponding GTFS schedule archives.
2. Parse `calendar.txt` and `calendar_dates.txt` (Exception Types 1 & 2) to resolve exact active `service_id` sets per date.
3. Calculate active scheduled trips and scheduled trip-hours per route and date using DuckDB[cite: 1].
4. Join scheduled daily service directly with observed ridership on `(route, date)`[cite: 1].

In [1]:
import os
import glob
import pandas as pd
import duckdb

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed")
GTFS_BASE_PATH = os.path.join("..", "data", "raw", "gtfs")

# Map date ranges to their catalog archive folders based on Handbook Section 5.3
ARCHIVE_WINDOWS = [
    ("2025-11-23", "2025-12-18", "mdb-389-202511230128"),
    ("2025-12-19", "2026-01-07", "mdb-389-202512190112"),
    ("2026-01-08", "2026-02-09", "mdb-389-202601080131"),
    ("2026-02-10", "2026-03-25", "mdb-389-202602100132"),
    ("2026-03-26", "2026-04-23", "mdb-389-202603260020"),
    ("2026-04-24", "2026-05-29", "mdb-389-202604240144"),
    ("2026-05-30", "2026-07-10", "mdb-389-202605300003"),
    ("2026-07-11", "2026-07-31", "mdb-389-202607110020"),
]

def get_archive_for_date(date_str):
    for start, end, archive in ARCHIVE_WINDOWS:
        if start <= date_str <= end:
            return archive
    return None

print("GTFS Archive mappings loaded successfully.")

GTFS Archive mappings loaded successfully.


## Helper Functions: Active Service Resolution & Duration Calculation
GTFS service days use `calendar.txt` weekly masks, modified by `calendar_dates.txt` (Exception 1 = Add Service, Exception 2 = Remove Service)[cite: 1].
GTFS arrival and departure times can exceed 24:00:00 (e.g., `25:10:00` for post-midnight trips), which we preserve[cite: 1].

In [2]:
def gtfs_time_to_seconds(time_str):
    """Convert GTFS HH:MM:SS string to total seconds, preserving >24 hour values."""
    parts = str(time_str).strip().split(':')
    if len(parts) != 3:
        return None
    try:
        hours, minutes, seconds = int(parts[0]), int(parts[1]), int(parts[2])
        return hours * 3600 + minutes * 60 + seconds
    except ValueError:
        return None

def get_active_service_ids(archive_dir, service_date):
    """Resolve active service_ids for a date considering weekly flags and exception dates."""
    date_dt = pd.to_datetime(service_date)
    day_name = date_dt.day_name().lower() # e.g. 'monday'
    date_yyyymmdd = date_dt.strftime('%Y%m%d')
    
    calendar_path = os.path.join(archive_dir, "calendar.txt")
    calendar_dates_path = os.path.join(archive_dir, "calendar_dates.txt")
    
    active_services = set()
    
    # 1. Evaluate weekly calendar flags
    if os.path.exists(calendar_path):
        cal_df = pd.read_csv(calendar_path, dtype=str)
        cal_df['start_date'] = cal_df['start_date'].str.strip()
        cal_df['end_date'] = cal_df['end_date'].str.strip()
        
        valid_cal = cal_df[
            (cal_df['start_date'] <= date_yyyymmdd) & 
            (cal_df['end_date'] >= date_yyyymmdd) & 
            (cal_df[day_name] == '1')
        ]
        active_services.update(valid_cal['service_id'].unique())
        
    # 2. Apply calendar_dates exceptions
    if os.path.exists(calendar_dates_path):
        exceptions_df = pd.read_csv(calendar_dates_path, dtype=str)
        exceptions_df['date'] = exceptions_df['date'].str.strip()
        
        date_exceptions = exceptions_df[exceptions_df['date'] == date_yyyymmdd]
        
        # Exception type 2: Remove service
        removed = set(date_exceptions[date_exceptions['exception_type'] == '2']['service_id'])
        active_services -= removed
        
        # Exception type 1: Add service
        added = set(date_exceptions[date_exceptions['exception_type'] == '1']['service_id'])
        active_services.update(added)
        
    return active_services

print("Service resolution functions compiled.")

Service resolution functions compiled.


## Process All Target Dates using DuckDB
Iterate over the 251 target dates (`2025-11-23` to `2026-07-31`), query GTFS tables using DuckDB, and aggregate scheduled trips and duration[cite: 1].

In [5]:
con = duckdb.connect(database=':memory:')

# Generate full date list
date_list = pd.date_range("2025-11-23", "2026-07-31").strftime('%Y-%m-%d').tolist()

scheduled_daily_records = []

print(f"Processing GTFS schedules across {len(date_list)} dates...")

for idx, current_date in enumerate(date_list):
    archive_folder = get_archive_for_date(current_date)
    if not archive_folder:
        continue
        
    archive_dir = os.path.join(GTFS_BASE_PATH, archive_folder)
    active_service_ids = get_active_service_ids(archive_dir, current_date)
    
    if not active_service_ids:
        continue
        
    # Convert paths to forward slashes for DuckDB
    trips_path = os.path.abspath(os.path.join(archive_dir, "trips.txt")).replace("\\", "/")
    stop_times_path = os.path.abspath(os.path.join(archive_dir, "stop_times.txt")).replace("\\", "/")
    routes_path = os.path.abspath(os.path.join(archive_dir, "routes.txt")).replace("\\", "/")
    
    service_id_list_str = ", ".join([f"'{s}'" for s in active_service_ids])
    
    # Explicit SQL syntax without aliasing conflicts
    query = f"""
        WITH bus_routes AS (
            SELECT route_id, route_short_name
            FROM read_csv_auto('{routes_path}', all_varchar=True)
            WHERE route_type = '3'
        ),
        active_trips AS (
            SELECT t.trip_id, r.route_short_name AS route
            FROM read_csv_auto('{trips_path}', all_varchar=True) AS t
            JOIN bus_routes AS r ON t.route_id = r.route_id
            WHERE t.service_id IN ({service_id_list_str})
        ),
        trip_durations AS (
            SELECT 
                a.route AS route_name,
                st.trip_id AS trip_id,
                MIN(st.departure_time) AS first_dep,
                MAX(st.arrival_time) AS last_arr
            FROM read_csv_auto('{stop_times_path}', all_varchar=True) AS st
            JOIN active_trips AS a ON st.trip_id = a.trip_id
            GROUP BY a.route, st.trip_id
        )
        SELECT 
            route_name AS route,
            COUNT(trip_id) AS scheduled_trips,
            LIST(first_dep) AS deps,
            LIST(last_arr) AS arrs
        FROM trip_durations
        GROUP BY route_name
    """
    
    res_df = con.execute(query).df()
    
    for _, row in res_df.iterrows():
        route = str(row['route']).strip()
        trips = row['scheduled_trips']
        deps = row['deps']
        arrs = row['arrs']
        
        total_seconds = 0
        valid_duration = True
        
        for dep, arr in zip(deps, arrs):
            t1 = gtfs_time_to_seconds(dep)
            t2 = gtfs_time_to_seconds(arr)
            if t1 is not None and t2 is not None and t2 >= t1:
                total_seconds += (t2 - t1)
            else:
                valid_duration = False
                
        service_hours = round(total_seconds / 3600.0, 2) if valid_duration else None
        
        scheduled_daily_records.append({
            "route": route,
            "service_date": current_date,
            "scheduled_trips": trips,
            "scheduled_service_hours": service_hours,
            "duration_complete": valid_duration,
            "source_snapshot": archive_folder
        })

scheduled_df = pd.DataFrame(scheduled_daily_records)
print(f"Schedule aggregation complete! Total schedule records: {len(scheduled_df):,}")

Processing GTFS schedules across 251 dates...
Schedule aggregation complete! Total schedule records: 28,398


## Join Schedules with Boardings & Validate Against Handbook Counts
Join scheduled daily service to cleaned boardings on `(route, date)` and compare matching statistics against the handbook targets[cite: 1].

In [8]:
# Load clean boardings
boardings_df = pd.read_parquet(os.path.join(PROCESSED_DATA_PATH, "clean_boardings_full.parquet"))
boardings_df['date_str'] = boardings_df['date'].dt.strftime('%Y-%m-%d')

# Exact inner join on route and date
matched_df = pd.merge(
    boardings_df,
    scheduled_df,
    left_on=['route', 'date_str'],
    right_on=['route', 'service_date'],
    how='inner'
)

# Calculate indicators
matched_df['boardings_per_scheduled_trip'] = matched_df['rides'] / matched_df['scheduled_trips']
matched_df['boardings_per_scheduled_service_hour'] = matched_df['rides'] / matched_df['scheduled_service_hours']

# Save matched dataset
output_matched_path = os.path.join(PROCESSED_DATA_PATH, "matched_demand_service.parquet")
output_scheduled_path = os.path.join(PROCESSED_DATA_PATH, "selected_daily_schedules.parquet")

matched_df.to_parquet(output_matched_path, index=False)
scheduled_df.to_parquet(output_scheduled_path, index=False)

# Compare to Handbook Targets
print("================ MATCHING VALIDATION RESULTS ================")
print(f"Calendar Dates Processed   : {scheduled_df['service_date'].nunique()} (Target: 251)")
print(f"Total Scheduled Records   : {len(scheduled_df):,} (Target: 28,398)")
print(f"Matched Demand/Service    : {len(matched_df):,} (Target: 28,374)")
print(f"Distinct Matched Routes   : {matched_df['route'].nunique()} (Target: 125)")
print("-------------------------------------------------------------")
print(f"Data saved to             : {PROCESSED_DATA_PATH}")
print("=============================================================")

================ MATCHING VALIDATION RESULTS ================
Calendar Dates Processed   : 251 (Target: 251)
Total Scheduled Records   : 28,398 (Target: 28,398)
Matched Demand/Service    : 28,374 (Target: 28,374)
Distinct Matched Routes   : 125 (Target: 125)
-------------------------------------------------------------
Data saved to             : ..\data\processed


# Step 3: Demand Feature Engineering & Chronological Dataset Splitting

## Objectives
1. Construct calendar features (day of week, month, weekend flag, daytype).
2. Compute exact historical target lags ($Lag_1, Lag_7, Lag_{14}, Lag_{21}, Lag_{28}$) and seasonal mean baselines.
3. Apply strict chronological partitions for Train, Validation, and Test sets[cite: 1].
4. Export versioned feature datasets for model training[cite: 1].

In [9]:
import os
import pandas as pd
import numpy as np

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed")

# Load 2-year clean boardings dataset
boardings_df = pd.read_parquet(os.path.join(PROCESSED_DATA_PATH, "clean_boardings_2yr.parquet"))

print(f"Loaded 2-Year Boardings Dataset: {len(boardings_df):,} rows")
print(f"Date Range: {boardings_df['date'].min().strftime('%Y-%m-%d')} to {boardings_df['date'].max().strftime('%Y-%m-%d')}")

Loaded 2-Year Boardings Dataset: 83,997 rows
Date Range: 2024-08-01 to 2026-07-31


## Generate Exact Date-Based Lag Features
To avoid Index-shift errors when dates are missing, we join `boardings_df` back onto itself with `date - lag_days`[cite: 1].

In [10]:
# Ensure explicit datetime sorting
boardings_df = boardings_df.sort_values(by=['route', 'date']).reset_index(drop=True)

# Add Calendar Features
boardings_df['day_of_week'] = boardings_df['date'].dt.dayofweek
boardings_df['month'] = boardings_df['date'].dt.month
boardings_df['is_weekend'] = boardings_df['day_of_week'].isin([5, 6]).astype(int)

# 1. Create Lag Lookup Table
lags = [1, 7, 14, 21, 28]

for lag in lags:
    # Target date for lag join
    lag_df = boardings_df[['route', 'date', 'rides']].copy()
    lag_df['join_date'] = lag_df['date'] + pd.Timedelta(days=lag)
    lag_df = lag_df.rename(columns={'rides': f'lag_{lag}'})
    
    # Merge lag back to primary table
    boardings_df = pd.merge(
        boardings_df,
        lag_df[['route', 'join_date', f'lag_{lag}']],
        left_on=['route', 'date'],
        right_on=['route', 'join_date'],
        how='left'
    ).drop(columns=['join_date'])

# 2. Compute Seasonal Baseline Mean across available 7, 14, 21, 28-day lags
seasonal_cols = ['lag_7', 'lag_14', 'lag_21', 'lag_28']

# Count non-null seasonal lags
boardings_df['available_seasonal_lags'] = boardings_df[seasonal_cols].notna().sum(axis=1)

# Compute mean of seasonal lags (requires at least 2 valid lags, else NaN)
boardings_df['seasonal_mean'] = np.where(
    boardings_df['available_seasonal_lags'] >= 2,
    boardings_df[seasonal_cols].mean(axis=1),
    np.nan
)

print("Exact date-based lag features created successfully.")
boardings_df[['route', 'date', 'rides', 'lag_1', 'lag_7', 'seasonal_mean']].head()

C:\Users\HP\AppData\Local\Temp\ipykernel_20000\3703500868.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  boardings_df['day_of_week'] = boardings_df['date'].dt.dayofweek


Exact date-based lag features created successfully.


,route,date,rides,lag_1,lag_7,seasonal_mean
0,1,2024-08-01,990,NaN,NaN,NaN
1,1,2024-08-02,627,990.0,NaN,NaN
2,1,2024-08-05,935,NaN,NaN,NaN
3,1,2024-08-06,1133,935.0,NaN,NaN
4,1,2024-08-07,1290,1133.0,NaN,NaN


## Generate Exact Date-Based Lag Features
To avoid Index-shift errors when dates are missing, we join `boardings_df` back onto itself with `date - lag_days`[cite: 1].

In [13]:
import numpy as np
import pandas as pd

# Ensure the required columns exist and dates use datetime format
required_cols = {"route", "date", "rides"}
missing_cols = required_cols - set(boardings_df.columns)
if missing_cols:
    raise KeyError(f"boardings_df is missing required columns: {sorted(missing_cols)}")

boardings_df = boardings_df.copy()
boardings_df["date"] = pd.to_datetime(boardings_df["date"], errors="coerce")

if boardings_df["date"].isna().any():
    raise ValueError("Some values in boardings_df['date'] could not be parsed as dates.")

# Remove lag columns from previous runs so rerunning this cell is safe
lag_days = [1, 7, 14, 21, 28]
columns_to_remove = [
    f"lag_{lag}" for lag in lag_days
    if f"lag_{lag}" in boardings_df.columns
]
if columns_to_remove:
    boardings_df = boardings_df.drop(columns=columns_to_remove)

# Sort observations by route and date
boardings_df = (
    boardings_df
    .sort_values(["route", "date"])
    .reset_index(drop=True)
)

# Add calendar features
boardings_df["day_of_week"] = boardings_df["date"].dt.dayofweek
boardings_df["month"] = boardings_df["date"].dt.month
boardings_df["is_weekend"] = boardings_df["day_of_week"].isin([5, 6]).astype(int)

# Create exact date-based lag features
for lag in lag_days:
    lag_df = boardings_df[["route", "date", "rides"]].copy()
    lag_df["date"] = lag_df["date"] + pd.Timedelta(days=lag)
    lag_df = lag_df.rename(columns={"rides": f"lag_{lag}"})

    boardings_df = boardings_df.merge(
        lag_df,
        on=["route", "date"],
        how="left",
        validate="one_to_one"
    )

# Calculate the seasonal baseline from available weekly lags
seasonal_cols = ["lag_7", "lag_14", "lag_21", "lag_28"]

boardings_df["available_seasonal_lags"] = (
    boardings_df[seasonal_cols].notna().sum(axis=1)
)

boardings_df["seasonal_mean"] = (
    boardings_df[seasonal_cols]
    .mean(axis=1)
    .where(boardings_df["available_seasonal_lags"] >= 2)
)

print("Exact date-based lag features created successfully.")

boardings_df[
    ["route", "date", "rides", "lag_1", "lag_7", "seasonal_mean"]
].head()

C:\Users\HP\AppData\Local\Temp\ipykernel_20000\3985719518.py:33: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  boardings_df["day_of_week"] = boardings_df["date"].dt.dayofweek


Exact date-based lag features created successfully.


,route,date,rides,lag_1,lag_7,seasonal_mean
0,1,2024-08-01,990,NaN,NaN,NaN
1,1,2024-08-02,627,990.0,NaN,NaN
2,1,2024-08-05,935,NaN,NaN,NaN
3,1,2024-08-06,1133,935.0,NaN,NaN
4,1,2024-08-07,1290,1133.0,NaN,NaN


## Apply Chronological Partitioning & Verify Row Counts
Partition data into Train, Validation, and Test sets based on the Handbook specs[cite: 1].

In [14]:
# Partition boundaries
train_df = boardings_df[(boardings_df['date'] >= '2024-08-01') & (boardings_df['date'] <= '2026-01-31')].copy()
val_df   = boardings_df[(boardings_df['date'] >= '2026-02-01') & (boardings_df['date'] <= '2026-04-30')].copy()
test_df  = boardings_df[(boardings_df['date'] >= '2026-05-01') & (boardings_df['date'] <= '2026-07-31')].copy()

# Add split labels
train_df['split'] = 'train'
val_df['split']   = 'val'
test_df['split']  = 'test'

full_features_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Export processed features
feature_parquet_path = os.path.join(PROCESSED_DATA_PATH, "demand_features_2yr.parquet")
full_features_df.to_parquet(feature_parquet_path, index=False)

# Validation Metrics Output
print("================ CHRONOLOGICAL SPLIT VALIDATION ================")
print(f"Train Rows (2024-08-01 to 2026-01-31) : {len(train_df):,}  (Target: 63,143)")
print(f"Val Rows   (2026-02-01 to 2026-04-30) : {len(val_df):,}  (Target: 10,246)")
print(f"Test Rows  (2026-05-01 to 2026-07-31) : {len(test_df):,}  (Target: 10,608)")
print("----------------------------------------------------------------")
print(f"Total Feature Rows Saved               : {len(full_features_df):,}  (Target: 83,997)")
print(f"Output saved to                        : {feature_parquet_path}")
print("================================================================")

================ CHRONOLOGICAL SPLIT VALIDATION ================
Train Rows (2024-08-01 to 2026-01-31) : 63,143  (Target: 63,143)
Val Rows   (2026-02-01 to 2026-04-30) : 10,246  (Target: 10,246)
Test Rows  (2026-05-01 to 2026-07-31) : 10,608  (Target: 10,608)
----------------------------------------------------------------
Total Feature Rows Saved               : 83,997  (Target: 83,997)
Output saved to                        : ..\data\processed\demand_features_2yr.parquet
